In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
# 🧩 라이브러리 임포트
import re
import pandas as pd
from tokenizers import Tokenizer, models, trainers, decoders
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import NFKC


# 클래스 정의

In [ ]:
# ✴️ 부정어 처리
class CustomNegationPreTokenizer:
    def pre_tokenize(self, pretok):
        text = pretok.normalized.get()
        text = re.sub(r'(절대|전혀|하나도|도저히)\s?(안|못)', r'\1안', text)
        text = re.sub(r'안 할 수 없다', '안할수없다', text)
        text = re.sub(r'(안|못)(돼|됐어|되냐|될걸|됨)', r'\1 되', text)
        text = re.sub(r'(말랬잖아)', r'말 았 잖아', text)
        text = re.sub(r'([가-힣]+)지\s?말(자|고|래|라|겠|까)?', r'\1 지 말 \2', text)
        text = re.sub(r'([가-힣]+)지\s?(않|못)([가-힣]*)', r'\1 지 \2 \3', text)
        text = re.sub(r'(안|못)([가-힣]+)', r'\1 \2', text)
        text = re.sub(r'([^ ]+)([이가]) 아니다', r'\1 \2 아니다', text)
        pretok.split(text.split())

# ✴️ 강조 접두사
class CustomEmphasisPreTokenizer:
    def pre_tokenize(self, pretok):
        text = pretok.normalized.get()
        exceptions = ["개는", "엄청나게", "완전무장", "되게하자", "존중", "핵심", "졸지마"]
        if any(word in text for word in exceptions):
            pretok.split(text.split()); return

        prefix = ['개','존','핵','겁','겁나','졸','엄청','되게','아주','진짜','완전','정말','무척','매우','굉장히']
        core = ['좋아','좋다','재밌','예쁘','멋지','무섭','귀엽','싫','슬프','기쁘','감동','편하','불편','놀랍','졸리']
        pattern2 = r'\b(' + '|'.join(prefix) + r')(' + '|'.join(prefix) + r')(' + '|'.join(core) + r'[가-힣]*)'
        pattern1 = r'\b(' + '|'.join(prefix) + r')(' + '|'.join(core) + r'[가-힣]+)'
        text = re.sub(pattern2, r'\1 \2 \3', text)
        text = re.sub(pattern1, r'\1 \2', text)
        pretok.split(text.split())

# ✴️ 띄어쓰기 오류
class CustomSpacingCorrectionPreTokenizer:
    def pre_tokenize(self, pretok):
        text = pretok.normalized.get()
        emphasis = ['완전','진짜','정말','되게','엄청','아주','너무']
        semantics = ['좋아','좋다','재밌','예쁘','멋지','무섭','귀엽','싫','슬프','기쁘','감동','편하','불편','놀랍','졸리',
                     '깨끗','시끄럽','복잡','간단','기분좋','행복','배고프','배부르','피곤','따뜻','추워','덥','아프','편안','힘들']
        endings = ['다','해','어','았어','었어','겠','네','한','한데']
        all_forms = semantics + [w + e for w in semantics for e in endings]
        pattern = '|'.join(sorted(set(all_forms), key=lambda x: -len(x)))
        p2 = r'\b(' + '|'.join(emphasis) + r')(' + '|'.join(emphasis) + r')(?=' + pattern + r')'
        p1 = r'\b(' + '|'.join(emphasis) + r')(?=' + pattern + r')'
        text = re.sub(p2, r'\1 \2', text)
        text = re.sub(p1, r'\1 ', text)
        pretok.split(text.split())

# ✴️ 슬랭/신조어
class CustomSlangPreTokenizer:
    def pre_tokenize(self, pretok):
        text = pretok.normalized.get()
        slang_map = {
            "ㄱㄱ":"고고", "ㅇㅋ":"오케이", "ㅈㅅ":"죄송", "ㄴㄴ":"노노", "ㅊㅋ":"축하",
            "ㄹㅇ":"리얼", "ㅂㅂ":"바이바이", "ㅁㅊ":"미친", "ㅋㅋㅋ":"크크크", "ㅎㅎ":"하하",
            "ㅠㅠ":"흑흑", "ㅇㅇ":"응응", "ㅅㅅ":"수고", "ㄷㄷ":"덜덜", "ㄴㄷ":"나도",
            "ㅂㄷㅂㄷ":"부들부들", "ㄷㅊ":"닥쳐", "ㅁㄹ":"몰라", "ㅊㅊ":"추천",
            "노잼":"재미없음", "꿀잼":"재미있음", "존맛":"정말 맛있다", "개이득":"큰 이득",
            "헐":"놀람", "짱":"최고", "쩐다":"대단하다", "쌉인정":"완전 동의",
            "갑분싸":"갑자기 분위기 싸해짐", "불소":"불타는 소통", "대깨문":"문재인 절대지지자"
        }
        for k, v in slang_map.items():
            text = re.sub(rf'\b{re.escape(k)}\b', v, text)
        pretok.split(text.split())

# ✴️ 의성어/의태어
class CustomOnomatopoeiaPreTokenizer:
    def pre_tokenize(self, pretok):
        text = pretok.normalized.get()
        text = re.sub(r'\b([가-힣]{1,2})\1{1,}\b', lambda m: ' '.join([m.group(1)] * (len(m.group(0)) // len(m.group(1)))), text)
        for word in ["쿵쿵", "탁탁", "딩동", "펑펑", "빙글빙글", "반짝반짝", "쩝쩝", "와르르", "두근두근", "드르렁", "으르렁", "쨍그랑", "철썩철썩", "휘파람", "쿨쿨", "깡총깡총"]:
            if word in text:
                parts = re.findall(r'[가-힣]{2}', word)
                text = text.replace(word, ' '.join(parts))
        pretok.split(text.split())

# ✴️ 느낌표/물음표 강조
class CustomPunctuationPreTokenizer:
    def pre_tokenize(self, pretok):
        text = pretok.normalized.get()
        text = re.sub(r'([!?]{2,})', r' \1 ', text)
        text = re.sub(r'([!?])$', r' \1', text)
        text = re.sub(r'(?<=[가-힣a-zA-Z0-9])([!?])(?=[가-힣a-zA-Z0-9])', r' \1 ', text)
        pretok.split(text.split())


# 데이터 불러오기 및 전처리

In [ ]:
# 📂 CSV 로딩
df = pd.read_csv("/content/drive/MyDrive/5차년도_2차.csv", encoding="cp949")
texts = df["발화문"].dropna().astype(str).tolist()

# 🔄 전체 전처리 적용 함수
def apply_all_custom_pretokenizers(text):
    dummy = type('DummyPretok', (), {})()
    dummy.normalized = dummy
    dummy.get = lambda: text

    for processor in [
        CustomSlangPreTokenizer(),
        CustomOnomatopoeiaPreTokenizer(),
        CustomPunctuationPreTokenizer(),
        CustomNegationPreTokenizer(),
        CustomEmphasisPreTokenizer(),
        CustomSpacingCorrectionPreTokenizer()
    ]:
        dummy.split = lambda splits: setattr(dummy, 'splits', splits)
        processor.pre_tokenize(dummy)
        text = " ".join(dummy.splits)

    return text

preprocessed_texts = [apply_all_custom_pretokenizers(t) for t in texts]


# BPE 토크나이저 학습 및 저장

In [ ]:
# ✨ BPE 모델 학습
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.normalizer = NFKC()
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(vocab_size=8000, show_progress=True,
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
)
tokenizer.train_from_iterator(preprocessed_texts, trainer=trainer)
tokenizer.decoder = decoders.BPEDecoder()
tokenizer.save("custom_tokenizer_all.json")


#결과 저장 및 미리보기

In [ ]:
with open("tokenized_all_texts.txt", "w", encoding="utf-8") as f:
    for sentence in preprocessed_texts:
        encoded = tokenizer.encode(sentence)
        f.write(f"원문: {sentence}\n")
        f.write(f"토큰: {encoded.tokens}\n")
        f.write(f"토큰 ID: {encoded.ids}\n\n")

# 👁 확인용 출력
with open("tokenized_all_texts.txt", "r", encoding="utf-8") as f:
    for _ in range(5):
        print(f.readline().strip())


원문: 놀람! 나 이벤트에 당첨 됐어.
토큰: ['놀람', '!', '나', '이벤트에', '당첨', '됐어', '.']
토큰 ID: [1149, 5, 151, 1179, 989, 1010, 9]

원문: 내가 좋아하는 인플루언서가 이벤트를 하더라고. 그래서 그냥 신청 한번 해봤지.


# 단어 빈도 집계

In [ ]:
from collections import Counter

# ✅ 1. 전처리된 문장들을 띄어쓰기 기준으로 분할하여 단어 리스트로 변환
all_words = []
for sentence in preprocessed_texts:
    all_words.extend(sentence.split())

# ✅ 2. 등장 빈도 계산
word_counts = Counter(all_words)

# ✅ 3. 자주 등장한 단어순으로 정렬된 목록 저장
output_file = "token_frequency_sorted.txt"
with open(output_file, "w", encoding="utf-8") as f:
    for word, count in word_counts.most_common():
        f.write(f"{word}\t{count}\n")

print(f"📁 저장 완료: {output_file}")


📁 저장 완료: token_frequency_sorted.txt
